# Campaign · Stage B — score every TSFM on C-MAPSS (one core runtime)

Stage B of the cross-TSFM C-MAPSS campaign. It runs on **ONE core runtime with NO backbones installed** — the five per-model embedding caches are already on Drive from the Stage-A notebooks (`chronos.ipynb`, `moment.ipynb`, `timesfm.ipynb`, `ttm.ipynb`, `moirai.ipynb`), so nothing here loads a TSFM. It:

1. reads the five caches from Drive and, per (dataset × model), trains the MLP head + the baselines (**including `catch22_gbm`**) and writes per-combo results CSVs + data-scaling / horizon figures (`run_campaign`, restartable);
2. assembles the **cross-TSFM success map** (win / tie / loss / hollow), a **combined cross-model data-scaling** view, and the **earliness + cost-curve** figures — all from the repo's existing tested functions.

**Before running:** run **all five** Stage-A notebooks first (one per GPU runtime) so every cache is on Drive. Point `DRIVE` at the SAME folder they used. A GPU here only speeds up head training — no backbone is needed.

In [ ]:
# 1) clone the campaign branch (shallow) — same pattern as notebooks/verify/*
%cd /content
!git clone --depth 1 --branch claude/c-mapss-colab-campaign-d5yut6 https://github.com/blozanod/Predictive-Maintenance-LSTM.git 2>/dev/null || echo "(already cloned — reusing)"
%cd /content/Predictive-Maintenance-LSTM
import sys; sys.path.insert(0, '/content/Predictive-Maintenance-LSTM')   # import src.* from the fresh clone

In [ ]:
# 2) install the CORE stack (requirements.txt) — no v2 backbones. This brings the
#    baseline deps (lightgbm, sktime, pycatch22 for catch22_gbm) + coral-pytorch (CORN).
#    Embeddings are cached, so chronos-forecasting is never imported here.
!pip install -r requirements.txt

In [ ]:
# 3) mount Google Drive — the SAME folder the Stage-A notebooks wrote the caches to
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4) the CANONICAL config — identical to every Stage-A notebook for the cache-key fields
#    (window / sensors / max_rul / pooling / context / condition_norm), so Stage B finds
#    the caches Stage A wrote. model_name is set per-combo by run_campaign(models=[...]).
from src.config import Config

DRIVE = '/content/drive/MyDrive/pdm_tsfm'          # SAME Drive folder Stage A wrote the caches to

config = Config(
    data_root='Data',                  # C-MAPSS committed in the repo clone
    cache_dir=f'{DRIVE}/cache',        # the five per-model caches from Stage A
    results_dir=f'{DRIVE}/results',
    tsfm_context_length=256,           # recorded FD001 winner (CHANGES.md §12)
    pooling='mean',
    head_features='emb+locscale',
)
config

## 1. Campaign — heads + baselines from the cached embeddings

`run_campaign` over **all five models × FD001–FD004**, stages `sweep → fairness → horizon → figures` (no `cache` stage — the embeddings already exist on Drive). Each (dataset × model) combo writes its own `results/<dataset>_<model>_results_v2.csv` and figures, and is restartable. `baseline_names` opts **`catch22_gbm`** into the roster (the hand-crafted-indicator foil) without changing `run_sweep`'s default list.

In [ ]:
import torch
from src.campaign import run_campaign
from src.models import EMBEDDERS

device = 'cuda' if torch.cuda.is_available() else 'cpu'   # heads/baselines only; NO backbone is loaded
MODELS = sorted(EMBEDDERS)                                 # all five registry keys

summary = run_campaign(
    config,
    models=MODELS,
    datasets=['FD001', 'FD002', 'FD003', 'FD004'],
    stages=['sweep', 'fairness', 'horizon', 'figures'],
    baseline_names=['predict_mean', 'gbm', 'minirocket', 'cnn', 'lstm', 'catch22_gbm'],
    device=device,
)
summary

## 2. Cross-TSFM success map (the headline object)

`scoring.success_map` reads every per-combo CSV, applies the per-cell win / tie / loss / hollow rule (primary metric `nasa_clipped`, strongest-competitor bar, predict-mean floor guard — RESEARCH_PLAN §8), and `plots.plot_success_map` renders the models × conditions heatmap, one panel per dataset.

In [ ]:
from pathlib import Path
from src import scoring
from src.plots import plot_success_map

results_glob = str(Path(config.results_dir) / '*_results_v2.csv')   # the per-combo CSVs
success = scoring.success_map(results_glob, config=config,
                              cell_fields=('dataset', 'n_units'))
print(f'success-map rows: {len(success)}')
if success:
    saved = plot_success_map(success, config.figures_dir(), prefix='cross_tsfm_', show=False)
    print('saved:', *[str(s) for s in saved], sep='\n  ')
else:
    print('no rows — did every Stage-A model cache land on Drive, and did the campaign run?')

## 3. Combined cross-model data-scaling

Concatenate the per-combo `*_results_v2.csv` into one frame so **all five TSFMs + the baselines share one curve per (dataset, metric)** in `plots.plot_data_scaling`. Baselines are re-run per combo, so their rows repeat across combo CSVs — deduped to one row per logical data point (the `<tag>_mlp` TSFM rows are unique per model, untouched).

In [ ]:
import glob
import pandas as pd
from src.plots import plot_data_scaling

files = sorted(glob.glob(str(Path(config.results_dir) / '*_results_v2.csv')))
combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
combined = combined.drop_duplicates(subset=['model', 'dataset', 'n_units', 'seed', 'loss'])
combined_csv = Path(config.results_dir) / 'cross_model_data_scaling.csv'   # not '*_results_v2.csv' -> not re-globbed
combined.to_csv(combined_csv, index=False)

saved = plot_data_scaling(combined_csv, config.figures_dir(), prefix='cross_model_', show=False)
print(f'combined {len(combined)} rows from {len(files)} combo CSVs')
print('saved:', *[str(s) for s in saved], sep='\n  ')

## 4. Earliness + cost curves ("too early is also bad")

`horizon.run_earliness` turns the per-cycle horizon predictions into the two-sided earliness histogram (`frac_late` vs `frac_early`) and the swept late:early cost curve (RESEARCH_PLAN §8). Concatenate the per-combo `*_horizon_predictions.csv` (deduping repeated baseline rows so the cost sums are not inflated), then score once over all models and render with `plots.plot_earliness` / `plots.plot_cost_curve`.

In [ ]:
from src.horizon import run_earliness
from src.plots import plot_earliness, plot_cost_curve

pred_files = sorted(glob.glob(str(Path(config.results_dir) / '*_horizon_predictions.csv')))
preds = pd.concat([pd.read_csv(f) for f in pred_files], ignore_index=True)
preds = preds.drop_duplicates(
    subset=['model', 'dataset', 'max_rul', 'n_units', 'seed', 'loss', 'unit'])
preds_csv = Path(config.results_dir) / 'cross_model_all_horizon_preds.csv'   # not re-globbed
preds.to_csv(preds_csv, index=False)

earliness_csv, cost_csv = run_earliness(config, preds_csv=preds_csv)
saved  = plot_earliness(earliness_csv, config.figures_dir(), prefix='cross_model_', show=False)
saved += plot_cost_curve(cost_csv, config.figures_dir(), prefix='cross_model_', show=False)
print('saved:', *[str(s) for s in saved], sep='\n  ')

## 5. Artifact summary

In [ ]:
figs = sorted(Path(config.figures_dir()).glob('*.png'))
print(f'{len(figs)} figures under {config.figures_dir()}:')
for f in figs:
    print('  ', f.name)
print(f'\nPer-combo results CSVs, the cross-model/cross-TSFM tables, and every figure '
      f'are under {config.results_dir} on Drive.')